In [0]:
import base64
import re
from datetime import datetime, timezone
from decimal import Decimal
from uuid import uuid4

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from pyspark.sql import functions as F


def ensure_text_widget(name: str, default_value: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value)


ensure_text_widget("catalog", "workspace")
ensure_text_widget("schema", "default")
ensure_text_widget("pipeline_run_id", "")
ensure_text_widget("publish_mode", "incremental")
ensure_text_widget(
    "s3_base_path",
    "s3://lucien-racine-aws-bucket/raw/databricks",
)
ensure_text_widget("secret_scope", "lakehouse")
ensure_text_widget("snowflake_stage", "RAW_S3_STAGE")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

provided_run_id = dbutils.widgets.get(
    "pipeline_run_id"
).strip()

PIPELINE_RUN_ID = provided_run_id or str(uuid4())

# Safe partition/path representation of the run ID.
RUN_ID = re.sub(
    r"[^A-Za-z0-9_-]",
    "_",
    PIPELINE_RUN_ID,
)

PUBLISH_MODE = (
    dbutils.widgets.get("publish_mode")
    .strip()
    .lower()
)

S3_BASE_PATH = (
    dbutils.widgets.get("s3_base_path")
    .strip()
    .rstrip("/")
)

SECRET_SCOPE = dbutils.widgets.get(
    "secret_scope"
).strip()

SNOWFLAKE_STAGE = (
    dbutils.widgets.get("snowflake_stage")
    .strip()
    .upper()
)


def validate_identifier(value: str, label: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid {label}: {value!r}")

    return value


validate_identifier(CATALOG, "catalog")
validate_identifier(SCHEMA, "schema")

if PUBLISH_MODE not in {"incremental", "full"}:
    raise ValueError(
        "publish_mode must be 'incremental' or 'full'"
    )

if not S3_BASE_PATH.startswith("s3://"):
    raise ValueError(
        "s3_base_path must begin with s3://"
    )

if not re.fullmatch(
    r"[A-Z_][A-Z0-9_$]*",
    SNOWFLAKE_STAGE,
):
    raise ValueError(
        f"Invalid Snowflake stage: {SNOWFLAKE_STAGE!r}"
    )

SILVER_SALES_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_sales"
)

SILVER_PRODUCTS_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_products"
)

SALES_S3_PATH = (
    f"{S3_BASE_PATH}/silver_sales/run_id={RUN_ID}/"
)

PRODUCTS_S3_PATH = (
    f"{S3_BASE_PATH}/silver_products/run_id={RUN_ID}/"
)

SALES_STAGE_PATH = (
    f"databricks/silver_sales/run_id={RUN_ID}/"
)

PRODUCTS_STAGE_PATH = (
    f"databricks/silver_products/run_id={RUN_ID}/"
)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Publish mode: {PUBLISH_MODE}")
print(f"Sales source: {SILVER_SALES_TABLE}")
print(f"Products source: {SILVER_PRODUCTS_TABLE}")
print(f"Sales output: {SALES_S3_PATH}")
print(f"Products output: {PRODUCTS_S3_PATH}")

In [0]:
import binascii


def get_secret(key: str) -> str:
    value = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key=key,
    )

    if not value:
        raise RuntimeError(
            f"Required secret is empty: {key}"
        )

    return value


encoded_private_key = get_secret(
    "snowflake-private-key-b64"
).strip()

try:
    private_key_pem = base64.b64decode(
        encoded_private_key,
        validate=True,
    )
except (binascii.Error, ValueError) as exc:
    raise RuntimeError(
        "The Snowflake private-key secret is not valid Base64."
    ) from exc

private_key_passphrase = get_secret(
    "snowflake-private-key-passphrase"
).encode("utf-8")

try:
    private_key_object = (
        serialization.load_pem_private_key(
            private_key_pem,
            password=private_key_passphrase,
        )
    )
except (TypeError, ValueError) as exc:
    raise RuntimeError(
        "Unable to decrypt the Snowflake RSA private key. "
        "Verify that the stored key and passphrase belong together."
    ) from exc

private_key_der = private_key_object.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

# Remove unnecessary plaintext key material from notebook variables.
del encoded_private_key
del private_key_pem
del private_key_passphrase
del private_key_object


def get_snowflake_connection():
    return snowflake.connector.connect(
        user=get_secret("snowflake-user").strip(),
        account=get_secret("snowflake-account").strip(),
        warehouse=get_secret(
            "snowflake-warehouse"
        ).strip(),
        database=get_secret(
            "snowflake-database"
        ).strip(),
        schema=get_secret(
            "snowflake-schema"
        ).strip(),
        role=get_secret("snowflake-role").strip(),
        private_key=private_key_der,
        login_timeout=30,
        network_timeout=120,
        client_session_keep_alive=False,
        session_parameters={
            "QUERY_TAG": (
                "the_lake_house:"
                f"{PIPELINE_RUN_ID}:"
                "publish_to_snowflake"
            )
        },
    )


print(
    "Snowflake RSA authentication configuration "
    "loaded successfully."
)

In [0]:
def validate_required_columns(
    table_name: str,
    required_columns: set[str],
) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required Silver table does not exist: {table_name}"
        )

    actual_columns = set(
        spark.table(table_name).columns
    )

    missing_columns = sorted(
        required_columns - actual_columns
    )

    if missing_columns:
        raise RuntimeError(
            f"{table_name} is missing required columns: "
            f"{missing_columns}"
        )


validate_required_columns(
    SILVER_SALES_TABLE,
    {
        "order_id",
        "sales_id",
        "product_id",
        "customer_id",
        "quantity",
        "unit_price",
        "total_amount",
        "ordered_at",
        "silver_record_hash",
        "transformation_run_id",
        "bronze_ingested_at",
    },
)

validate_required_columns(
    SILVER_PRODUCTS_TABLE,
    {
        "product_id",
        "product_name",
        "product_description",
        "price",
        "category",
        "image_url",
        "rating",
        "rating_count",
        "silver_record_hash",
        "transformation_run_id",
        "bronze_ingested_at",
    },
)

snowflake_database = get_secret(
    "snowflake-database"
).strip().upper()

snowflake_schema = get_secret(
    "snowflake-schema"
).strip().upper()

snowflake_identifiers = {
    "database": snowflake_database,
    "schema": snowflake_schema,
    "stage": SNOWFLAKE_STAGE,
}

for label, value in snowflake_identifiers.items():
    if not re.fullmatch(
        r"[A-Z_][A-Z0-9_$]*",
        value,
    ):
        raise ValueError(
            f"Unsafe Snowflake {label} identifier: {value!r}"
        )

run_suffix = re.sub(
    r"[^A-Z0-9_]",
    "_",
    RUN_ID.upper(),
)[:40]

RAW_SALES_TABLE = (
    f"{snowflake_database}."
    f"{snowflake_schema}.RAW_SALES"
)

RAW_PRODUCTS_TABLE = (
    f"{snowflake_database}."
    f"{snowflake_schema}.RAW_PRODUCTS"
)

SALES_LOAD_TABLE = (
    f"{snowflake_database}.{snowflake_schema}."
    f"RAW_SALES_LOAD_{run_suffix}"
)

PRODUCTS_LOAD_TABLE = (
    f"{snowflake_database}.{snowflake_schema}."
    f"RAW_PRODUCTS_LOAD_{run_suffix}"
)

AUDIT_TABLE = (
    f"{snowflake_database}.{snowflake_schema}."
    "PIPELINE_RUN_AUDIT"
)

QUALIFIED_STAGE = (
    f"{snowflake_database}.{snowflake_schema}."
    f"{SNOWFLAKE_STAGE}"
)

connection = get_snowflake_connection()
cursor = None

try:
    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT
            CURRENT_ROLE(),
            CURRENT_DATABASE(),
            CURRENT_SCHEMA(),
            CURRENT_WAREHOUSE()
        """
    )

    (
        active_role,
        active_database,
        active_schema,
        active_warehouse,
    ) = cursor.fetchone()

finally:
    if cursor is not None:
        cursor.close()

    connection.close()

print("Silver source schemas validated.")
print("Snowflake connection validated.")
print(f"Role: {active_role}")
print(f"Database: {active_database}")
print(f"Schema: {active_schema}")
print(f"Warehouse: {active_warehouse}")

In [0]:
silver_sales_source_df = spark.table(
    SILVER_SALES_TABLE
)

silver_products_source_df = spark.table(
    SILVER_PRODUCTS_TABLE
)

if PUBLISH_MODE == "incremental":
    sales_publish_source_df = (
        silver_sales_source_df
        .filter(
            F.col("transformation_run_id")
            == F.lit(PIPELINE_RUN_ID)
        )
    )

    products_publish_source_df = (
        silver_products_source_df
        .filter(
            F.col("transformation_run_id")
            == F.lit(PIPELINE_RUN_ID)
        )
    )
else:
    sales_publish_source_df = silver_sales_source_df
    products_publish_source_df = silver_products_source_df


sales_export_df = (
    sales_publish_source_df
    .select(
        F.col("sales_id")
        .cast("long")
        .alias("SALES_ID"),
        F.col("product_id")
        .cast("long")
        .alias("PRODUCT_ID"),
        F.col("customer_id")
        .cast("long")
        .alias("CUST_ID"),
        F.col("quantity")
        .cast("integer")
        .alias("QTY"),
        F.col("unit_price")
        .cast("decimal(12, 2)")
        .alias("UNIT_PRICE"),
        F.col("total_amount")
        .cast("decimal(14, 2)")
        .alias("TOTAL_AMT"),
        F.col("ordered_at").alias("SALE_DATE"),
        F.col("bronze_ingested_at").alias("UPDATED_AT"),
        F.current_timestamp().alias("DBT_UPDATED_AT"),
        F.col("silver_record_hash").alias("RECORD_HASH"),
        F.col("transformation_run_id").alias(
            "PIPELINE_RUN_ID"
        ),
    )
)

products_export_df = (
    products_publish_source_df
    .select(
        F.col("product_id")
        .cast("long")
        .alias("ID"),
        F.col("product_name").alias("TITLE"),
        F.col("price")
        .cast("decimal(12, 2)")
        .alias("PRICE"),
        F.col("product_description").alias(
            "DESCRIPTION"
        ),
        F.col("category").alias("CATEGORY"),
        F.col("image_url").alias("IMAGE"),
        F.concat(
            F.lit("{'rate': "),
            F.col("rating").cast("string"),
            F.lit(", 'count': "),
            F.col("rating_count").cast("string"),
            F.lit("}"),
        ).alias("RATING"),
        F.col("bronze_ingested_at").alias("UPDATED_AT"),
        F.current_timestamp().alias("INGESTION_TS"),
        F.col("silver_record_hash").alias("RECORD_HASH"),
        F.col("transformation_run_id").alias(
            "PIPELINE_RUN_ID"
        ),
    )
)

sales_change_count = sales_export_df.count()
products_change_count = products_export_df.count()

HAS_SALES_CHANGES = sales_change_count > 0
HAS_PRODUCT_CHANGES = products_change_count > 0
HAS_CHANGES = HAS_SALES_CHANGES or HAS_PRODUCT_CHANGES

if HAS_SALES_CHANGES:
    (
        sales_export_df.write
        .format("parquet")
        .option("compression", "snappy")
        .option("maxRecordsPerFile", 1_000_000)
        .mode("overwrite")
        .save(SALES_S3_PATH)
    )

if HAS_PRODUCT_CHANGES:
    (
        products_export_df.write
        .format("parquet")
        .option("compression", "snappy")
        .option("maxRecordsPerFile", 1_000_000)
        .mode("overwrite")
        .save(PRODUCTS_S3_PATH)
    )

print(f"Publish mode: {PUBLISH_MODE}")
print(f"Sales rows selected: {sales_change_count:,}")
print(f"Product rows selected: {products_change_count:,}")

if not HAS_CHANGES:
    print(
        "No Silver changes were produced by this pipeline run. "
        "No Parquet files were written."
    )
else:
    if HAS_SALES_CHANGES:
        print(f"Sales output: {SALES_S3_PATH}")

    if HAS_PRODUCT_CHANGES:
        print(f"Products output: {PRODUCTS_S3_PATH}")

In [0]:
connection = get_snowflake_connection()
cursor = None

try:
    cursor = connection.cursor()

    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {RAW_SALES_TABLE} (
            SALES_ID NUMBER(38, 0),
            PRODUCT_ID NUMBER(38, 0),
            CUST_ID NUMBER(38, 0),
            QTY INTEGER,
            UNIT_PRICE DECIMAL(12, 2),
            TOTAL_AMT DECIMAL(14, 2),
            SALE_DATE TIMESTAMP_NTZ,
            UPDATED_AT TIMESTAMP_NTZ,
            DBT_UPDATED_AT TIMESTAMP_NTZ,
            RECORD_HASH STRING,
            PIPELINE_RUN_ID STRING
        )
        """
    )

    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {RAW_PRODUCTS_TABLE} (
            ID NUMBER(38, 0),
            TITLE STRING,
            PRICE DECIMAL(12, 2),
            DESCRIPTION STRING,
            CATEGORY STRING,
            IMAGE STRING,
            RATING STRING,
            UPDATED_AT TIMESTAMP_NTZ,
            INGESTION_TS TIMESTAMP_NTZ,
            RECORD_HASH STRING,
            PIPELINE_RUN_ID STRING
        )
        """
    )

    # Add lineage columns to tables created by the older pipeline.
    cursor.execute(
        f"""
        ALTER TABLE {RAW_SALES_TABLE}
        ADD COLUMN IF NOT EXISTS RECORD_HASH STRING
        """
    )

    cursor.execute(
        f"""
        ALTER TABLE {RAW_SALES_TABLE}
        ADD COLUMN IF NOT EXISTS PIPELINE_RUN_ID STRING
        """
    )

    cursor.execute(
        f"""
        ALTER TABLE {RAW_PRODUCTS_TABLE}
        ADD COLUMN IF NOT EXISTS RECORD_HASH STRING
        """
    )

    cursor.execute(
        f"""
        ALTER TABLE {RAW_PRODUCTS_TABLE}
        ADD COLUMN IF NOT EXISTS PIPELINE_RUN_ID STRING
        """
    )

    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
            RUN_ID STRING,
            STATUS STRING,
            PUBLISH_MODE STRING,
            SALES_ROWS INTEGER,
            PRODUCT_ROWS INTEGER,
            TOTAL_REVENUE DECIMAL(18, 2),
            SALES_S3_PATH STRING,
            PRODUCTS_S3_PATH STRING,
            ERROR_MESSAGE STRING,
            STARTED_AT TIMESTAMP_NTZ,
            COMPLETED_AT TIMESTAMP_NTZ
        )
        """
    )

    # Upgrade the existing audit table without destroying its history.
    audit_columns = [
        ("PUBLISH_MODE", "STRING"),
        ("ERROR_MESSAGE", "STRING"),
        ("STARTED_AT", "TIMESTAMP_NTZ"),
    ]

    for column_name, data_type in audit_columns:
        cursor.execute(
            f"""
            ALTER TABLE {AUDIT_TABLE}
            ADD COLUMN IF NOT EXISTS
                {column_name} {data_type}
            """
        )

finally:
    if cursor is not None:
        cursor.close()

    connection.close()

print("Snowflake target tables are ready.")
print("Snowflake audit table is ready.")

In [0]:
if HAS_SALES_CHANGES:
    expected_revenue_value = (
        sales_export_df
        .agg(
            F.sum("TOTAL_AMT").alias("TOTAL_REVENUE")
        )
        .first()["TOTAL_REVENUE"]
    )

    expected_revenue = Decimal(
        str(expected_revenue_value or 0)
    )
else:
    expected_revenue = Decimal("0")


def write_audit_status(
    status: str,
    error_message: str | None = None,
) -> None:
    completed_statuses = {
        "SUCCEEDED",
        "FAILED",
        "NO_CHANGES",
    }

    is_complete = status in completed_statuses

    audit_connection = get_snowflake_connection()
    audit_cursor = None

    try:
        audit_cursor = audit_connection.cursor()

        audit_cursor.execute(
            f"""
            MERGE INTO {AUDIT_TABLE} AS target
            USING (
                SELECT
                    %s AS RUN_ID,
                    %s AS STATUS,
                    %s AS PUBLISH_MODE,
                    %s AS SALES_ROWS,
                    %s AS PRODUCT_ROWS,
                    %s AS TOTAL_REVENUE,
                    %s AS SALES_S3_PATH,
                    %s AS PRODUCTS_S3_PATH,
                    %s AS ERROR_MESSAGE,
                    %s AS IS_COMPLETE,
                    CURRENT_TIMESTAMP() AS EVENT_TIME
            ) AS source
                ON target.RUN_ID = source.RUN_ID

            WHEN MATCHED THEN UPDATE SET
                STATUS = source.STATUS,
                PUBLISH_MODE = source.PUBLISH_MODE,
                SALES_ROWS = source.SALES_ROWS,
                PRODUCT_ROWS = source.PRODUCT_ROWS,
                TOTAL_REVENUE = source.TOTAL_REVENUE,
                SALES_S3_PATH = source.SALES_S3_PATH,
                PRODUCTS_S3_PATH = source.PRODUCTS_S3_PATH,
                ERROR_MESSAGE = source.ERROR_MESSAGE,
                STARTED_AT = COALESCE(
                    target.STARTED_AT,
                    source.EVENT_TIME
                ),
                COMPLETED_AT = CASE
                    WHEN source.IS_COMPLETE
                    THEN source.EVENT_TIME
                    ELSE NULL
                END

            WHEN NOT MATCHED THEN INSERT (
                RUN_ID,
                STATUS,
                PUBLISH_MODE,
                SALES_ROWS,
                PRODUCT_ROWS,
                TOTAL_REVENUE,
                SALES_S3_PATH,
                PRODUCTS_S3_PATH,
                ERROR_MESSAGE,
                STARTED_AT,
                COMPLETED_AT
            )
            VALUES (
                source.RUN_ID,
                source.STATUS,
                source.PUBLISH_MODE,
                source.SALES_ROWS,
                source.PRODUCT_ROWS,
                source.TOTAL_REVENUE,
                source.SALES_S3_PATH,
                source.PRODUCTS_S3_PATH,
                source.ERROR_MESSAGE,
                source.EVENT_TIME,
                CASE
                    WHEN source.IS_COMPLETE
                    THEN source.EVENT_TIME
                    ELSE NULL
                END
            )
            """,
            (
                PIPELINE_RUN_ID,
                status,
                PUBLISH_MODE,
                sales_change_count,
                products_change_count,
                expected_revenue,
                SALES_S3_PATH,
                PRODUCTS_S3_PATH,
                error_message,
                is_complete,
            ),
        )

    finally:
        if audit_cursor is not None:
            audit_cursor.close()

        audit_connection.close()


def copy_parquet_into_table(
    cursor,
    target_table: str,
    stage_path: str,
) -> None:
    cursor.execute(
        f"""
        COPY INTO {target_table}
        FROM @{QUALIFIED_STAGE}/{stage_path}
        FILE_FORMAT = (
            TYPE = PARQUET
        )
        PATTERN = '.*[.]parquet'
        MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
        ON_ERROR = ABORT_STATEMENT
        FORCE = TRUE
        PURGE = FALSE
        """
    )


if not HAS_CHANGES:
    write_audit_status("NO_CHANGES")
    PUBLISH_STATUS = "NO_CHANGES"

    print(
        "No new or changed Silver records were available. "
        "Snowflake production tables were not modified."
    )

else:
    write_audit_status("RUNNING")

    connection = get_snowflake_connection()
    cursor = None
    transaction_started = False
    publish_succeeded = False

    try:
        cursor = connection.cursor()

        if HAS_SALES_CHANGES:
            cursor.execute(
                f"""
                CREATE OR REPLACE TRANSIENT TABLE
                    {SALES_LOAD_TABLE}
                LIKE {RAW_SALES_TABLE}
                """
            )

            copy_parquet_into_table(
                cursor,
                SALES_LOAD_TABLE,
                SALES_STAGE_PATH,
            )

            cursor.execute(
                f"""
                SELECT
                    COUNT(*) AS ROW_COUNT,
                    COALESCE(
                        SUM(TOTAL_AMT),
                        0
                    ) AS TOTAL_REVENUE
                FROM {SALES_LOAD_TABLE}
                """
            )

            (
                loaded_sales_count,
                loaded_revenue,
            ) = cursor.fetchone()

            if loaded_sales_count != sales_change_count:
                raise RuntimeError(
                    "Sales staging reconciliation failed: "
                    f"expected={sales_change_count}, "
                    f"loaded={loaded_sales_count}"
                )

            revenue_difference = abs(
                Decimal(str(loaded_revenue))
                - expected_revenue
            )

            if revenue_difference > Decimal("0.01"):
                raise RuntimeError(
                    "Sales revenue reconciliation failed: "
                    f"difference={revenue_difference}"
                )

        if HAS_PRODUCT_CHANGES:
            cursor.execute(
                f"""
                CREATE OR REPLACE TRANSIENT TABLE
                    {PRODUCTS_LOAD_TABLE}
                LIKE {RAW_PRODUCTS_TABLE}
                """
            )

            copy_parquet_into_table(
                cursor,
                PRODUCTS_LOAD_TABLE,
                PRODUCTS_STAGE_PATH,
            )

            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM {PRODUCTS_LOAD_TABLE}
                """
            )

            loaded_product_count = cursor.fetchone()[0]

            if loaded_product_count != products_change_count:
                raise RuntimeError(
                    "Product staging reconciliation failed: "
                    f"expected={products_change_count}, "
                    f"loaded={loaded_product_count}"
                )

        cursor.execute("BEGIN")
        transaction_started = True

        if HAS_SALES_CHANGES:
            cursor.execute(
                f"""
                MERGE INTO {RAW_SALES_TABLE} AS target
                USING {SALES_LOAD_TABLE} AS source
                    ON target.SALES_ID = source.SALES_ID
                   AND target.PRODUCT_ID = source.PRODUCT_ID

                WHEN MATCHED
                 AND NOT EQUAL_NULL(
                     target.RECORD_HASH,
                     source.RECORD_HASH
                 )
                THEN UPDATE SET
                    CUST_ID = source.CUST_ID,
                    QTY = source.QTY,
                    UNIT_PRICE = source.UNIT_PRICE,
                    TOTAL_AMT = source.TOTAL_AMT,
                    SALE_DATE = source.SALE_DATE,
                    UPDATED_AT = source.UPDATED_AT,
                    DBT_UPDATED_AT = source.DBT_UPDATED_AT,
                    RECORD_HASH = source.RECORD_HASH,
                    PIPELINE_RUN_ID =
                        source.PIPELINE_RUN_ID

                WHEN NOT MATCHED THEN INSERT (
                    SALES_ID,
                    PRODUCT_ID,
                    CUST_ID,
                    QTY,
                    UNIT_PRICE,
                    TOTAL_AMT,
                    SALE_DATE,
                    UPDATED_AT,
                    DBT_UPDATED_AT,
                    RECORD_HASH,
                    PIPELINE_RUN_ID
                )
                VALUES (
                    source.SALES_ID,
                    source.PRODUCT_ID,
                    source.CUST_ID,
                    source.QTY,
                    source.UNIT_PRICE,
                    source.TOTAL_AMT,
                    source.SALE_DATE,
                    source.UPDATED_AT,
                    source.DBT_UPDATED_AT,
                    source.RECORD_HASH,
                    source.PIPELINE_RUN_ID
                )
                """
            )

            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM {SALES_LOAD_TABLE} AS source
                LEFT JOIN {RAW_SALES_TABLE} AS target
                    ON target.SALES_ID = source.SALES_ID
                   AND target.PRODUCT_ID = source.PRODUCT_ID
                WHERE target.SALES_ID IS NULL
                   OR NOT EQUAL_NULL(
                       target.RECORD_HASH,
                       source.RECORD_HASH
                   )
                """
            )

            unmatched_sales = cursor.fetchone()[0]

            if unmatched_sales != 0:
                raise RuntimeError(
                    "Post-merge sales reconciliation failed: "
                    f"unmatched={unmatched_sales}"
                )

        if HAS_PRODUCT_CHANGES:
            cursor.execute(
                f"""
                MERGE INTO {RAW_PRODUCTS_TABLE} AS target
                USING {PRODUCTS_LOAD_TABLE} AS source
                    ON target.ID = source.ID

                WHEN MATCHED
                 AND NOT EQUAL_NULL(
                     target.RECORD_HASH,
                     source.RECORD_HASH
                 )
                THEN UPDATE SET
                    TITLE = source.TITLE,
                    PRICE = source.PRICE,
                    DESCRIPTION = source.DESCRIPTION,
                    CATEGORY = source.CATEGORY,
                    IMAGE = source.IMAGE,
                    RATING = source.RATING,
                    UPDATED_AT = source.UPDATED_AT,
                    INGESTION_TS = source.INGESTION_TS,
                    RECORD_HASH = source.RECORD_HASH,
                    PIPELINE_RUN_ID =
                        source.PIPELINE_RUN_ID

                WHEN NOT MATCHED THEN INSERT (
                    ID,
                    TITLE,
                    PRICE,
                    DESCRIPTION,
                    CATEGORY,
                    IMAGE,
                    RATING,
                    UPDATED_AT,
                    INGESTION_TS,
                    RECORD_HASH,
                    PIPELINE_RUN_ID
                )
                VALUES (
                    source.ID,
                    source.TITLE,
                    source.PRICE,
                    source.DESCRIPTION,
                    source.CATEGORY,
                    source.IMAGE,
                    source.RATING,
                    source.UPDATED_AT,
                    source.INGESTION_TS,
                    source.RECORD_HASH,
                    source.PIPELINE_RUN_ID
                )
                """
            )

            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM {PRODUCTS_LOAD_TABLE} AS source
                LEFT JOIN {RAW_PRODUCTS_TABLE} AS target
                    ON target.ID = source.ID
                WHERE target.ID IS NULL
                   OR NOT EQUAL_NULL(
                       target.RECORD_HASH,
                       source.RECORD_HASH
                   )
                """
            )

            unmatched_products = cursor.fetchone()[0]

            if unmatched_products != 0:
                raise RuntimeError(
                    "Post-merge product reconciliation failed: "
                    f"unmatched={unmatched_products}"
                )

        connection.commit()
        transaction_started = False
        publish_succeeded = True
        PUBLISH_STATUS = "SUCCEEDED"

        write_audit_status("SUCCEEDED")

        print("Incremental Snowflake publish succeeded.")
        print(f"Sales rows merged: {sales_change_count:,}")
        print(
            f"Product rows merged: "
            f"{products_change_count:,}"
        )

    except Exception as exc:
        if transaction_started:
            connection.rollback()

        PUBLISH_STATUS = "FAILED"

        audit_error = (
            f"{type(exc).__name__}: {str(exc)}"
        )[:4000]

        try:
            write_audit_status(
                "FAILED",
                error_message=audit_error,
            )
        except Exception as audit_exc:
            print(
                "WARNING: Failure status could not be "
                f"written to Snowflake: {audit_exc}"
            )

        raise

    finally:
        if cursor is not None:
            if publish_succeeded:
                if HAS_SALES_CHANGES:
                    cursor.execute(
                        f"""
                        DROP TABLE IF EXISTS
                            {SALES_LOAD_TABLE}
                        """
                    )

                if HAS_PRODUCT_CHANGES:
                    cursor.execute(
                        f"""
                        DROP TABLE IF EXISTS
                            {PRODUCTS_LOAD_TABLE}
                        """
                    )

            cursor.close()

        connection.close()

In [0]:
connection = get_snowflake_connection()
cursor = None

try:
    cursor = connection.cursor()

    cursor.execute(
        f"""
        SELECT
            STATUS,
            PUBLISH_MODE,
            SALES_ROWS,
            PRODUCT_ROWS,
            TOTAL_REVENUE,
            ERROR_MESSAGE,
            STARTED_AT,
            COMPLETED_AT
        FROM {AUDIT_TABLE}
        WHERE RUN_ID = %s
        ORDER BY COALESCE(
            COMPLETED_AT,
            STARTED_AT
        ) DESC
        LIMIT 1
        """,
        (PIPELINE_RUN_ID,),
    )

    audit_result = cursor.fetchone()

    if audit_result is None:
        raise RuntimeError(
            "No Snowflake audit record was created for "
            f"run {PIPELINE_RUN_ID}"
        )

    (
        audited_status,
        audited_mode,
        audited_sales_rows,
        audited_product_rows,
        audited_revenue,
        audited_error,
        audited_started_at,
        audited_completed_at,
    ) = audit_result

    if audited_status not in {
        "SUCCEEDED",
        "NO_CHANGES",
    }:
        raise RuntimeError(
            "Snowflake publishing did not finish successfully: "
            f"status={audited_status}, "
            f"error={audited_error}"
        )

    cursor.execute(
        f"""
        SELECT
            COUNT(*) AS ROW_COUNT,
            COUNT_IF(RECORD_HASH IS NULL)
                AS MISSING_RECORD_HASHES
        FROM {RAW_SALES_TABLE}
        """
    )

    (
        snowflake_sales_rows,
        sales_missing_hashes,
    ) = cursor.fetchone()

    cursor.execute(
        f"""
        SELECT
            COUNT(*) AS ROW_COUNT,
            COUNT_IF(RECORD_HASH IS NULL)
                AS MISSING_RECORD_HASHES
        FROM {RAW_PRODUCTS_TABLE}
        """
    )

    (
        snowflake_product_rows,
        products_missing_hashes,
    ) = cursor.fetchone()

finally:
    if cursor is not None:
        cursor.close()

    connection.close()

print("Snowflake publishing audit validated.")
print(f"Run ID: {PIPELINE_RUN_ID}")
print(f"Status: {audited_status}")
print(f"Mode: {audited_mode}")
print(f"Published sales rows: {audited_sales_rows:,}")
print(f"Published product rows: {audited_product_rows:,}")
print(f"Snowflake sales rows: {snowflake_sales_rows:,}")
print(f"Snowflake product rows: {snowflake_product_rows:,}")
print(
    "Legacy sales rows without record hashes: "
    f"{sales_missing_hashes:,}"
)
print(
    "Legacy product rows without record hashes: "
    f"{products_missing_hashes:,}"
)